# 04 — LOYO Cross-Validation + Peak Evaluation (ngưỡng 4‰)

**Tự chứa**: các cell 1–7 dưới đây là BẢN SAO NGUYÊN VĂN của `03_scenario3` (import, config, `load_station`/`split_years`/`load_split`, `SalinityDataset`/`build_dataset`, `train_model`/`evaluate`, `plot_*`, `BEST_PARAMS`). Chạy lần lượt từ trên xuống, không cần file 03.

Phần MỚI (từ mục A): Leave-One-Year-Out, peak metrics ngưỡng cố định 4‰, và vòng lặp đa model.

**Lưu ý:** năm êm (2025) thường không có điểm >4‰ → peak = NaN (đúng). R² ở năm đỉnh (2020, 2021) sẽ thấp hơn — đó là bức tranh thật.

In [1]:
import sys, os
import itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from models import build_model
from patchTST_Base import PatchTST_BASE
from patchTST_CT import CT_PatchTST
from patchTST_Intra import PatchTST_IntraTCN
import joblib
warnings.filterwarnings("ignore")

import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [14]:
# ── Paths ──────────────────────────────────────────────────────────────
DATA_DIR   = Path("data/final_dataset")
OUTPUT_DIR = Path("outputs")
PLOT_DIR   = OUTPUT_DIR / "plots"
MODEL_DIR  = OUTPUT_DIR / "models"
for d in [PLOT_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

# ── Splits ─────────────────────────────────────────────────────────────
TRAIN_YEARS = [2020,2021,2022]
VAL_YEARS   = [2023]
TEST_YEARS  = [2025]

STATIONS     = ["BenLuc", "CauNoi", "TanAn"]
FEATURE_COLS = ["wind_speed", "temp", "total_precipitation"]
#"PATCHTST_BASE","CT_PATCHTST",
# MODEL_NAMES  = ["PATCHTST_INTRATCN","PATCHTST_BASE","CT_PATCHTST"]
MODEL_NAMES  = ["PATCHTST_INTRATCN"]
LOOKBACKS    = [168]   # bước × 2h → 12h, 24h, 48h
# HORIZONS     = [3,6,12]
HORIZONS     = [3]

# ── Hyperparameters ────────────────────────────────────────────────────
EPOCHS     = 100
BATCH_SIZE = 32
LR         = 1e-3
PATIENCE   = 15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


Device: cuda


In [15]:
def load_station(name: str) -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / f"{name}_clean.csv",
                       parse_dates=["Time"])

def split_years(df: pd.DataFrame, years: list) -> pd.DataFrame:
    return df[df["Time"].dt.year.isin(years)].copy()

def load_split(stations: list, target: str):
    """Load & split all needed station data."""
    data = {s: load_station(s) for s in set(stations + [target])}
    splits = {}
    for label, yrs in [("train", TRAIN_YEARS), ("val", VAL_YEARS), ("test", TEST_YEARS)]:
        splits[label] = {s: split_years(data[s], yrs) for s in data}
    return splits


In [16]:
class SalinityDataset(Dataset):
    def __init__(self, X, y, F):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.F = torch.tensor(F, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i], self.F[i]

def make_windows(features, target, lookback, horizon):
    Xs, ys, Fs = [], [], []
    for i in range(len(features) - lookback - horizon + 1):
        Xs.append(features[i:i+lookback])
        ys.append(target[i+lookback:i+lookback+horizon])
        Fs.append(features[i+lookback:i+lookback+horizon, -6:])  # tương lai đã biết
    if not Xs:
        return (np.empty((0, lookback, features.shape[1]), np.float32),
                np.empty((0, horizon), np.float32),
                np.empty((0, horizon, 6), np.float32))
    return np.array(Xs, np.float32), np.array(ys, np.float32), np.array(Fs, np.float32)
def make_windows_gapaware(df, X_sc, y_sc, lookback, horizon, step=pd.Timedelta("2h")):
    seg_id = (df["Time"].diff() != step).cumsum()
    Xa, ya, Fa = [], [], []
    for _, idx in df.groupby(seg_id).indices.items():
        Xw, yw, Fw = make_windows(X_sc[idx], y_sc[idx], lookback, horizon)
        if len(Xw): Xa.append(Xw); ya.append(yw); Fa.append(Fw)
    if not Xa:   # không có đoạn nào đủ dài → trả mảng rỗng đúng shape
        return (np.empty((0, lookback, X_sc.shape[1]), dtype=np.float32),
                np.empty((0, horizon), dtype=np.float32))
    return np.concatenate(Xa), np.concatenate(ya), np.concatenate(Fa)
def add_tide_features(df, time_col="Time"):
    df_feat = df.copy()
    df_feat[time_col] = pd.to_datetime(df_feat[time_col])

    hours = df_feat[time_col].dt.hour + df_feat[time_col].dt.minute / 60.0

    df_feat['hour_sin'] = np.sin(2*np.pi*hours/24.0)
    df_feat['hour_cos'] = np.cos(2*np.pi*hours/24.0)

    reference_new_moon = pd.to_datetime("2020-01-25 04:42:00")

    delta_day = (df_feat[time_col] - reference_new_moon).dt.total_seconds()/(24*3600)

    lunar_cycle = 29.530588
    lunar_phase = delta_day%lunar_cycle

    df_feat['lunar_sin'] = np.sin(2 * np.pi * lunar_phase / lunar_cycle)
    df_feat['lunar_cos'] = np.cos(2* np.pi * lunar_phase / lunar_cycle)

    day_of_year = df_feat[time_col].dt.dayofyear
    day_in_year = 365.2425
    df_feat['year_sin'] = np.sin(2 * np.pi * day_of_year/day_in_year)
    df_feat['year_cos'] = np.cos(2 * np.pi * day_of_year/day_in_year)

    return df_feat
def build_dataset(splits, input_stations, target_station,
                  lookback, horizon, scaler_X=None, scaler_y=None, fit=False):
    """
    Merge input features, build sliding windows per split.
    Returns X_arr, y_arr, scaler_X, scaler_y
    """
    sal_cols    = [f"Salinity_{s}" for s in input_stations]
    target_col  = f"Salinity_{target_station}"
    #################################
    feature_cols = sal_cols

    all_X, all_y = [], []
    for split_name in ["train", "val", "test"] if fit else []:
        pass  # handled below

    def _merge_split(split_dfs):
        dfs = [split_dfs[s][["Time", f"Salinity_{s}"]].copy() for s in input_stations]
        merged = dfs[0]
        for d in dfs[1:]:
            merged = pd.merge(merged, d, on="Time", how="inner")
        meteo = split_dfs[input_stations[0]][["Time"] + FEATURE_COLS]
        merged = pd.merge(merged, meteo, on="Time", how="inner")
        if target_col not in merged.columns:
            tgt = split_dfs[target_station][["Time", target_col]]
            merged = pd.merge(merged, tgt, on="Time", how="inner")
        merged = merged.sort_values("Time").reset_index(drop=True)
        
        merged = add_tide_features(merged)
        
        return merged

    # Fit scalers on train
    train_merged = _merge_split(splits["train"])
    X_raw = train_merged[feature_cols].values.astype(np.float32)
    y_raw = train_merged[[target_col]].values.astype(np.float32)
    if fit:
        scaler_X = StandardScaler(); scaler_X.fit(X_raw)
        scaler_y = StandardScaler(); scaler_y.fit(y_raw)

    result = {}
    for split_name, split_dfs in splits.items():
        merged = _merge_split(split_dfs)
        X_sc = scaler_X.transform(merged[feature_cols].values.astype(np.float32))
        y_sc = scaler_y.transform(merged[[target_col]].values.astype(np.float32)).flatten()
        Xa, ya, Fa = [], [], []
        for yr in sorted(merged["Time"].dt.year.unique()):
            m = merged["Time"].dt.year == yr
            Xw, yw, Fw = make_windows(X_sc[m], y_sc[m], lookback, horizon)
            Xa.append(Xw); ya.append(yw), Fa.append(Fw)
        result[split_name] = (np.concatenate(Xa), np.concatenate(ya), np.concatenate(Fa))

    return result, scaler_X, scaler_y


In [17]:
def train_model(model, tr_loader, val_loader,lr=LR):
    model = model.to(DEVICE)
    criterion = nn.MSELoss()
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    best_val, pat_cnt, best_state = float("inf"), 0, None
    history = {"train": [], "val": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss = 0.0
        for Xb, yb, Fb in tr_loader:
            Xb, yb, Fb = Xb.to(DEVICE), yb.to(DEVICE), Fb.to(DEVICE)
            opt.zero_grad()
            out = model(Xb, Fb) if getattr(model, "use_future", False) else model(Xb)
            loss = criterion(out, yb)
            if getattr(model, "aux_loss", None) is not None:
                loss = loss + model.aux_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_loss += loss.item() * len(Xb)
        tr_loss /= len(tr_loader.dataset)

        model.eval()
        vl_loss = 0.0
        with torch.no_grad():
            for Xb, yb, Fb in val_loader:
                Xb, yb, Fb = Xb.to(DEVICE), yb.to(DEVICE), Fb.to(DEVICE)
                out = model(Xb, Fb) if getattr(model, "use_future", False) else model(Xb)
                vl_loss += criterion(out, yb).item() * len(Xb)
        vl_loss /= len(val_loader.dataset)

        history["train"].append(tr_loss)
        history["val"].append(vl_loss)
        sch.step(vl_loss)

        if vl_loss < best_val:
            best_val  = vl_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat_cnt   = 0
        else:
            pat_cnt += 1
            if pat_cnt >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, history

def evaluate(model, loader, scaler_y):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb, Fb in loader:                                   # SỬA 1: thêm Fb
            Xb, Fb = Xb.to(DEVICE), Fb.to(DEVICE)                   # SỬA 2
            out = model(Xb, Fb) if getattr(model, "use_future", False) else model(Xb)   # SỬA 3
            preds.append(out.cpu().numpy())
            trues.append(yb.numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    pi = scaler_y.inverse_transform(preds.reshape(-1,1)).reshape(preds.shape)
    ti = scaler_y.inverse_transform(trues.reshape(-1,1)).reshape(trues.shape)
    flat_p, flat_t = pi.flatten(), ti.flatten()
    return {
        "mse": float(mean_squared_error(flat_t, flat_p)),
        "mae" : float(mean_absolute_error(flat_t, flat_p)),
        "r2"  : float(r2_score(flat_t, flat_p)),
        "preds": pi, "trues": ti,
    }


In [18]:
def plot_loss(history, tag):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(history["train"], label="Train")
    ax.plot(history["val"],   label="Validation")
    ax.set_title(f"Loss — {tag}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
    ax.legend(); fig.tight_layout()
    fig.savefig(PLOT_DIR / f"loss_{tag}.png", dpi=110)
    plt.close(fig)

def plot_pred(preds, trues, tag):
    """
    2 subplot:
      - Trên: trung bình ngày toàn bộ test set → thấy xu hướng xâm nhập mặn
      - Dưới: chi tiết 7 ngày đầu theo từng bước 2h → thấy dao động thủy triều
    preds/trues shape: (N_samples, horizon)
    """
    # Lấy bước cuối của mỗi window → chuỗi dự báo liên tục
    p = preds[:, -1]   # (N,)
    t = trues[:, -1]   # (N,)
    n = len(p)

    # ── Panel trên: trung bình theo ngày ─────────────────────────
    # Mỗi bước = 2h → 12 bước/ngày
    steps_per_day = 12
    n_days = n // steps_per_day
    p_daily = p[:n_days*steps_per_day].reshape(n_days, steps_per_day).mean(axis=1)
    t_daily = t[:n_days*steps_per_day].reshape(n_days, steps_per_day).mean(axis=1)

    # ── Panel dưới: 7 ngày đầu (84 bước) ─────────────────────────
    n_detail = min(84, n)   # 7 ngày × 12 bước/ngày

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7),
                                   gridspec_kw={"height_ratios": [1.2, 1]})
    fig.suptitle(tag, fontsize=10, color="#424242")

    # Panel 1 — xu hướng theo ngày
    ax1.plot(t_daily, label="Thực tế",  lw=1.4, color="#1565C0")
    ax1.plot(p_daily, label="Dự báo",   lw=1.2, color="#E53935", alpha=0.85)
    ax1.set_title("Xu hướng toàn tập Test (trung bình ngày)", fontsize=10)
    ax1.set_xlabel("Ngày"); ax1.set_ylabel("Độ mặn TB (g/l)")
    ax1.spines[["top","right"]].set_visible(False)
    ax1.yaxis.grid(True, color="#EEEEEE", linewidth=0.8)
    ax1.set_axisbelow(True)
    ax1.legend(fontsize=9)

    # Panel 2 — chi tiết 7 ngày
    xticks = list(range(0, n_detail, steps_per_day))
    xlabels = [f"Ngày {i//steps_per_day+1}" for i in xticks]
    ax2.plot(t[:n_detail], label="Thực tế",  lw=1.2, color="#1565C0")
    ax2.plot(p[:n_detail], label="Dự báo",   lw=1.0, color="#E53935", alpha=0.85)
    ax2.set_title("Chi tiết 7 ngày đầu (bước 2h — thấy dao động thủy triều)", fontsize=10)
    ax2.set_xlabel("Thời gian"); ax2.set_ylabel("Độ mặn (g/l)")
    ax2.set_xticks(xticks); ax2.set_xticklabels(xlabels, fontsize=8)
    ax2.spines[["top","right"]].set_visible(False)
    ax2.yaxis.grid(True, color="#EEEEEE", linewidth=0.8)
    ax2.set_axisbelow(True)
    ax2.legend(fontsize=9)

    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"pred_{tag}.png", dpi=110, bbox_inches="tight")
    plt.close(fig)

In [19]:
import json
with open(r"C:\Users\Qminh\Desktop\Paper2026\SaltIntrusion\src_code\outputs\gridsearch\plots\best_params_2.json") as f:
    BEST_PARAMS = json.load(f)

#Thêm tham số cho mô hình mới (chỉnh tay các con số ở đây)
BEST_PARAMS["PATCHTST_BASE"] = {
        "hidden_size":   128,     # d_model
        "num_layers":    3,
        "learning_rate": 1e-4,    # KHÔNG dùng 1e-3 — dễ phân kỳ
        "batch_size":    32,
        "patch_len":     6,       # 12h ~ 1 con nước
        "stride":        6,
    }

BEST_PARAMS["CT_PATCHTST"] = dict(BEST_PARAMS["PATCHTST_BASE"])


BEST_PARAMS["PATCHTST_INTRATCN"] = {
    "hidden_size": 256, "num_layers": 4, "learning_rate": 1e-4,
    "batch_size": 32, "patch_len": 6, "stride": 6,
    "inter_mode": "mixer",     # đổi "attn" <-> "mixer" để ablation
    "inter_layers": 1,
}

print(BEST_PARAMS)

{'RNN': {'hidden_size': 256, 'num_layers': 3, 'dropout': 0.0, 'lr': 0.001, 'batch_size': 32, 'learning_rate': 0.0001}, 'LSTM': {'hidden_size': 256, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.001, 'batch_size': 32, 'learning_rate': 0.001}, 'GRU': {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.001, 'batch_size': 32, 'learning_rate': 0.0005}, 'CNN_LSTM': {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 64, 'learning_rate': 0.001}, 'PATCHTST_BASE': {'hidden_size': 128, 'num_layers': 3, 'learning_rate': 0.0001, 'batch_size': 32, 'patch_len': 6, 'stride': 6}, 'CT_PATCHTST': {'hidden_size': 128, 'num_layers': 3, 'learning_rate': 0.0001, 'batch_size': 32, 'patch_len': 6, 'stride': 6}, 'PATCHTST_INTRATCN': {'hidden_size': 256, 'num_layers': 4, 'learning_rate': 0.0001, 'batch_size': 32, 'patch_len': 6, 'stride': 6, 'inter_mode': 'mixer', 'inter_layers': 1}}


## A. Cấu hình LOYO + peak threshold

In [22]:
ALL_YEARS      = [2020, 2021, 2022, 2023, 2025]
PEAK_THRESHOLD = 4.0            # ngưỡng mặn nguy hiểm (‰), cố định cho mọi năm
LOYO_LOOKBACK  = 168           # 336h
LOYO_HORIZON   = 3             # 6h ; đổi 6->12h, 12->24h
LOYO_SEEDS     = [42, 43, 44]
LOYO_MODELS    = ['PATCHTST_BASE', 'CT_PATCHTST', 'PATCHTST_INTRATCN']

# Fallback BEST_PARAMS cho LSTM/GRU nếu file JSON chưa có (chỉnh cho khớp build_model của bạn)
for _m in ['LSTM', 'GRU']:
    if _m not in BEST_PARAMS:
        BEST_PARAMS[_m] = {'hidden_size': 128, 'num_layers': 2,
                           'learning_rate': 1e-3, 'batch_size': 32}
print('Models:', LOYO_MODELS)
print('Threshold:', PEAK_THRESHOLD, '‰ | horizon:', LOYO_HORIZON)

Models: ['PATCHTST_BASE', 'CT_PATCHTST', 'PATCHTST_INTRATCN']
Threshold: 4.0 ‰ | horizon: 3


## B. Peak metrics (ngưỡng cố định) + LOYO folds

In [23]:
def peak_metrics_fixed(preds, trues, threshold=PEAK_THRESHOLD):
    p = np.asarray(preds).flatten(); t = np.asarray(trues).flatten()
    mask = t >= threshold
    n = int(mask.sum())
    if n == 0:
        return {'peak_rmse': np.nan, 'peak_mae': np.nan, 'peak_bias': np.nan, 'n_peak': 0}
    err = p[mask] - t[mask]
    return {'peak_rmse': float(np.sqrt((err**2).mean())),
            'peak_mae' : float(np.abs(err).mean()),
            'peak_bias': float(err.mean()),   # <0 = HỤT đỉnh
            'n_peak': n}

def exceedance_recall(preds, trues, threshold=PEAK_THRESHOLD):
    p = np.asarray(preds).flatten() >= threshold
    t = np.asarray(trues).flatten() >= threshold
    tp = int((p & t).sum()); fn = int((~p & t).sum()); fp = int((p & ~t).sum())
    rec  = tp/(tp+fn) if (tp+fn) else np.nan
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    return {'recall': rec, 'precision': prec, 'n_true_exceed': int(t.sum())}

def make_loyo_folds(years=ALL_YEARS):
    folds = []
    for test_y in years:
        rest = [y for y in years if y != test_y]
        folds.append({'test': [test_y], 'val': [rest[-1]], 'train': rest[:-1]})
    return folds

for f in make_loyo_folds():
    print(f"test={f['test']}  val={f['val']}  train={f['train']}")

test=[2020]  val=[2025]  train=[2021, 2022, 2023]
test=[2021]  val=[2025]  train=[2020, 2022, 2023]
test=[2022]  val=[2025]  train=[2020, 2021, 2023]
test=[2023]  val=[2025]  train=[2020, 2021, 2022]
test=[2025]  val=[2023]  train=[2020, 2021, 2022]


## C. `load_split` theo năm tuỳ ý + factory tạo model
Factory sao y logic khởi tạo model trong `run_experiment` của bạn (cùng tham số).

In [24]:
def load_split_years(stations, target, train_years, val_years, test_years):
    data = {s: load_station(s) for s in set(stations + [target])}
    splits = {}
    for label, yrs in [('train', train_years), ('val', val_years), ('test', test_years)]:
        splits[label] = {s: split_years(data[s], yrs) for s in data}
    return splits

def build_loyo_model(model_name, input_size, lookback, horizon):
    p = BEST_PARAMS[model_name]
    if model_name == 'PATCHTST_BASE':
        return PatchTST_BASE(use_future=p.get('use_future', False),
            input_size=input_size, context_len=lookback, horizon=horizon,
            d_model=p['hidden_size'], n_layers=p['num_layers'],
            patch_len=p.get('patch_len',6), stride=p.get('stride',6), target_idx=0)
    if model_name == 'CT_PATCHTST':
        return CT_PatchTST(input_size=input_size, context_len=lookback, horizon=horizon,
            d_model=p['hidden_size'], n_layers=p['num_layers'],
            patch_len=p.get('patch_len',6), stride=p.get('stride',6), target_idx=0,
            n_channel_layers=p.get('n_channel_layers',1))
    if model_name == 'PATCHTST_INTRATCN':
        return PatchTST_IntraTCN(input_size=input_size, context_len=lookback, horizon=horizon,
            d_model=p['hidden_size'], n_layers=p['num_layers'],
            patch_len=p.get('patch_len',6), stride=p.get('stride',6), target_idx=0,
            inter_layers=p.get('inter_layers', 2), inter_mode=p.get('inter_mode','mixer'),
            intra_mode=p.get('intra_mode','attn'), pool_mode=p.get('pool_mode','flatten'),
            channel_mode=p.get('channel_mode','mixed'), longrange=p.get('longrange',True),
            n_queries=4)
    # LSTM, GRU, ... qua build_model (giống nhánh else của run_experiment)
    return build_model(model_name, input_size, horizon,
                       hidden_size=p['hidden_size'], num_layers=p['num_layers'])

## D. Vòng lặp LOYO chính
Mỗi (model × năm test × seed): train → evaluate (thang gốc ‰) → overall + peak. Lưu CSV dần.

In [27]:
RESULTS_CSV = OUTPUT_DIR / f'loyo_results_hz{LOYO_HORIZON}_trans.csv'
rows = []
folds = make_loyo_folds()
for model_name in LOYO_MODELS:
    p = BEST_PARAMS[model_name]
    for fold in folds:
        for seed in LOYO_SEEDS:
            set_seed(seed)
            splits = load_split_years(STATIONS, 'BenLuc',
                                      fold['train'], fold['val'], fold['test'])
            ds, scX, scY = build_dataset(splits, STATIONS, 'BenLuc',
                                         LOYO_LOOKBACK, LOYO_HORIZON, fit=True)
            if any(len(ds[k][0]) == 0 for k in ['train','val','test']):
                print(f'SKIP {model_name} test={fold["test"]} seed={seed} (cửa sổ rỗng)'); continue
            g = torch.Generator(); g.manual_seed(seed)
            bs = p.get('batch_size', 32)
            tr  = DataLoader(SalinityDataset(*ds['train']), batch_size=bs, shuffle=True, generator=g)
            va  = DataLoader(SalinityDataset(*ds['val']),   batch_size=bs)
            te  = DataLoader(SalinityDataset(*ds['test']),  batch_size=bs)

            model = build_loyo_model(model_name, ds['train'][0].shape[2], LOYO_LOOKBACK, LOYO_HORIZON)
            model, _ = train_model(model, tr, va, lr=p['learning_rate'])
            m = evaluate(model, te, scY)          # m['mse'], m['mae'], m['r2'], m['preds'], m['trues']

            pk  = peak_metrics_fixed(m['preds'], m['trues'])
            det = exceedance_recall(m['preds'], m['trues'])
            rows.append({
                'model': model_name, 'test_year': fold['test'][0], 'seed': seed,
                'R2': round(m['r2'],4), 'MAE': round(m['mae'],4),
                'RMSE': round(np.sqrt(m['mse']),4), 'MSE': round(m['mse'],4),
                'peak_rmse': round(pk['peak_rmse'],4) if pk['n_peak'] else np.nan,
                'peak_bias': round(pk['peak_bias'],4) if pk['n_peak'] else np.nan,
                'n_peak': pk['n_peak'],
                'recall': round(det['recall'],4) if det['n_true_exceed'] else np.nan,
                'n_exceed': det['n_true_exceed'],
            })
            pr = (f"R2={m['r2']:.3f}" +
                  (f" peak_rmse={pk['peak_rmse']:.3f} peak_bias={pk['peak_bias']:+.3f}" if pk['n_peak'] else ' (no peak)'))
            print(f"{model_name:18} test={fold['test'][0]} seed={seed} | {pr} n_peak={pk['n_peak']}")
            pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)

df = pd.DataFrame(rows); print('\nĐã lưu:', RESULTS_CSV)

PATCHTST_BASE      test=2020 seed=42 | R2=0.851 peak_rmse=1.276 peak_bias=-0.081 n_peak=2279
PATCHTST_BASE      test=2020 seed=43 | R2=0.859 peak_rmse=1.236 peak_bias=-0.048 n_peak=2279
PATCHTST_BASE      test=2020 seed=44 | R2=0.857 peak_rmse=1.243 peak_bias=-0.053 n_peak=2279
PATCHTST_BASE      test=2021 seed=42 | R2=0.927 peak_rmse=0.669 peak_bias=-0.282 n_peak=423
PATCHTST_BASE      test=2021 seed=43 | R2=0.924 peak_rmse=0.668 peak_bias=-0.202 n_peak=423
PATCHTST_BASE      test=2021 seed=44 | R2=0.929 peak_rmse=0.634 peak_bias=-0.246 n_peak=423
PATCHTST_BASE      test=2022 seed=42 | R2=0.900 (no peak) n_peak=0
PATCHTST_BASE      test=2022 seed=43 | R2=0.899 (no peak) n_peak=0
PATCHTST_BASE      test=2022 seed=44 | R2=0.891 (no peak) n_peak=0
PATCHTST_BASE      test=2023 seed=42 | R2=0.698 (no peak) n_peak=0
PATCHTST_BASE      test=2023 seed=43 | R2=0.673 (no peak) n_peak=0
PATCHTST_BASE      test=2023 seed=44 | R2=0.689 (no peak) n_peak=0
PATCHTST_BASE      test=2025 seed=42 | R2=0

## E. Tổng hợp kết quả

In [12]:
df = pd.read_csv(RESULTS_CSV)
print('=== R2 (mean qua seed) theo năm test ===')
print(df.groupby(['model','test_year'])['R2'].mean().unstack().round(4), '\n')

peak_df = df[df['n_peak'] > 0]
print('=== peak_rmse (mean) — chỉ năm có đỉnh >4‰ ===')
print(peak_df.groupby(['model','test_year'])['peak_rmse'].mean().unstack().round(4), '\n')

print('=== peak_bias (mean) — âm = HỤT đỉnh ===')
print(peak_df.groupby(['model','test_year'])['peak_bias'].mean().unstack().round(4), '\n')

print('=== recall vượt ngưỡng (mean) — năm có đỉnh ===')
print(peak_df.groupby(['model','test_year'])['recall'].mean().unstack().round(4))

=== R2 (mean qua seed) theo năm test ===
test_year            2020    2021    2022    2023    2025
model                                                    
CT_PATCHTST        0.8585  0.9159  0.8675  0.7025  0.8270
GRU                0.6618  0.9422  0.9170  0.8106  0.9084
LSTM               0.6389  0.9352  0.9235  0.8082  0.9151
PATCHTST_BASE      0.8558  0.9265  0.8968  0.6866  0.8293
PATCHTST_INTRATCN  0.8469  0.9199  0.8840  0.7387  0.8559 

=== peak_rmse (mean) — chỉ năm có đỉnh >4‰ ===
test_year            2020    2021
model                            
CT_PATCHTST        1.2453  0.7160
GRU                2.0034  0.5415
LSTM               2.0744  0.5778
PATCHTST_BASE      1.2516  0.6568
PATCHTST_INTRATCN  1.2996  0.6362 

=== peak_bias (mean) — âm = HỤT đỉnh ===
test_year            2020    2021
model                            
CT_PATCHTST       -0.0420 -0.1992
GRU               -1.1512  0.0269
LSTM              -1.1645  0.0197
PATCHTST_BASE     -0.0608 -0.2433
PATCHTST_INTRATCN -